In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix

import os 
from dotenv import load_dotenv

# loading the env variables from .env file
load_dotenv()

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)



print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
documents = [
    # ---------------- Economics ----------------
    "Global stock markets surged today as investors welcomed strong corporate earnings, and "
    "analysts said the stock market rally reflects growing confidence in the economy.",

    "Consumer prices continued to climb this month, and economists warned that rising inflation "
    "and higher prices across the economy could weaken household spending power.",

    "The central bank left interest rates unchanged, but policymakers said future interest rate "
    "decisions will depend on economic growth and inflation data in the coming months.",

    "A weakening currency and falling exports have raised concerns among economists about the "
    "overall economy, with slower economic growth expected across major markets next year.",

    # ---------------- Entertainment ----------------
    "The blockbuster superhero movie shattered box office records over the weekend, and fans "
    "praised the film for its stunning visual effects and gripping movie storyline.",

    "The pop star's surprise album release sent fans into a frenzy, as the singer's new songs "
    "topped music charts and became an instant fan favorite this week.",

    "Critics praised the new drama series for its compelling performances, calling it the "
    "must-watch television show of the season and a favorite among streaming audiences.",

    # ---------------- Politics ----------------
    "Voters headed to the polls in record numbers for the closely watched election, with both "
    "candidates campaigning hard for votes ahead of the final election results.",

    "The opposition candidate launched a nationwide election campaign, promising government "
    "reforms and appealing to voters ahead of next month's national election.",

    "Parliament remained deadlocked after lawmakers failed to pass the government's proposed "
    "budget bill, as opposition lawmakers in government continued to block the vote.",
]

true_categories = [
    "Economics", "Economics", "Economics", "Economics",
    "Entertainment", "Entertainment", "Entertainment",
    "Politics", "Politics", "Politics",
]

# defining a frame for the documents and actual category
df = pd.DataFrame({"document": documents, "true_category": true_categories})
df

,document,true_category
0,Global stock markets surged today as investors...,Economics
1,"Consumer prices continued to climb this month,...",Economics
2,The central bank left interest rates unchanged...,Economics
3,A weakening currency and falling exports have ...,Economics
4,The blockbuster superhero movie shattered box ...,Entertainment
5,The pop star's surprise album release sent fan...,Entertainment
6,Critics praised the new drama series for its c...,Entertainment
7,Voters headed to the polls in record numbers f...,Politics
8,The opposition candidate launched a nationwide...,Politics
9,Parliament remained deadlocked after lawmakers...,Politics


In [3]:
# calculating TF-IDF of each word by using scikit-learn's TfidfVectorizer

# removing stopwords
vectorizer = TfidfVectorizer(stop_words='english')
# vectorizing the documents
X_tfidf = vectorizer.fit_transform(df['document'])

print("TF-IDF matrix shape:", X_tfidf.shape)
print(f"({X_tfidf.shape[0]} documents x {X_tfidf.shape[1]} unique vocabulary words)")

# looking few of the vocabulary words TF-IDF learned
feature_names = vectorizer.get_feature_names_out()
print("\nSample vocabulary words:", list(feature_names[:15]))

TF-IDF matrix shape: (10, 137)
(10 documents x 137 unique vocabulary words)

Sample vocabulary words: ['ahead', 'album', 'analysts', 'appealing', 'audiences', 'bank', 'block', 'blockbuster', 'box', 'budget', 'calling', 'campaign', 'campaigning', 'candidate', 'candidates']


In [4]:
# clustering the documents with K-Means 

# three categories as "Economics", "Entertainment", "Politics"
K = 3
kmeans = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10)
# fitting the vectorized documents in the model
cluster_labels = kmeans.fit_predict(X_tfidf)

df['cluster'] = cluster_labels
df


,document,true_category,cluster
0,Global stock markets surged today as investors...,Economics,2
1,"Consumer prices continued to climb this month,...",Economics,2
2,The central bank left interest rates unchanged...,Economics,2
3,A weakening currency and falling exports have ...,Economics,2
4,The blockbuster superhero movie shattered box ...,Entertainment,1
5,The pop star's surprise album release sent fan...,Entertainment,1
6,Critics praised the new drama series for its c...,Entertainment,1
7,Voters headed to the polls in record numbers f...,Politics,0
8,The opposition candidate launched a nationwide...,Politics,0
9,Parliament remained deadlocked after lawmakers...,Politics,0


In [5]:
# checking the clusters

def top_terms_per_cluster(kmeans_model, vectorizer, n_terms=8):
    # finding top TF-IDF (at top) near each centroid
    order_centroids = kmeans_model.cluster_centers_.argsort()[:, ::-1]
    # getting the names
    terms = vectorizer.get_feature_names_out()
    
    
    for i in range(kmeans_model.n_clusters):
        top_words = [terms[ind] for ind in order_centroids[i, :n_terms]]
        print(f"Cluster {i}: {', '.join(top_words)}")

top_terms_per_cluster(kmeans, vectorizer)

Cluster 0: election, government, lawmakers, voters, ahead, opposition, promising, reforms
Cluster 1: movie, new, favorite, praised, fans, television, performances, streaming
Cluster 2: economy, prices, stock, growth, economic, economists, inflation, markets


In [6]:
# mapping each cluster (0,1,2) to each category

def build_cluster_to_category_map(df):
    mapping = {}
    for cluster_id in sorted(df['cluster'].unique()):
        subset = df[df['cluster'] == cluster_id]
        majority_category = subset['true_category'].mode()[0]
        mapping[cluster_id] = majority_category
    return mapping

cluster_to_category = build_cluster_to_category_map(df)
print("Cluster -> Category mapping:", cluster_to_category)

df['predicted_category'] = df['cluster'].map(cluster_to_category)
df[['document', 'true_category', 'cluster', 'predicted_category']]

Cluster -> Category mapping: {np.int32(0): 'Politics', np.int32(1): 'Entertainment', np.int32(2): 'Economics'}


,document,true_category,cluster,predicted_category
0,Global stock markets surged today as investors...,Economics,2,Economics
1,"Consumer prices continued to climb this month,...",Economics,2,Economics
2,The central bank left interest rates unchanged...,Economics,2,Economics
3,A weakening currency and falling exports have ...,Economics,2,Economics
4,The blockbuster superhero movie shattered box ...,Entertainment,1,Entertainment
5,The pop star's surprise album release sent fan...,Entertainment,1,Entertainment
6,Critics praised the new drama series for its c...,Entertainment,1,Entertainment
7,Voters headed to the polls in record numbers f...,Politics,0,Politics
8,The opposition candidate launched a nationwide...,Politics,0,Politics
9,Parliament remained deadlocked after lawmakers...,Politics,0,Politics


In [7]:
from pymongo import MongoClient


In [8]:
def launch_db(db_name, mongo_url):
    
        
    client = MongoClient(local_mongo_url)
    db = client[db_name]

    # for storing clustered data to 
    clustered_docs = db["clustered_docs"]

    print("Connected:", db.name, "| collections:", db.list_collection_names())
    
    return db, clustered_docs

In [9]:
db_name = "vertical_search_engine_cw"
local_mongo_url = "mongodb://localhost:27017"

# for local mongodb 
db, clustered_docs = launch_db(db_name, local_mongo_url)


# for cloud mongodb 
# db, clustered_docs = launch_db(db_name=os.getenv("CLOUD_DB_NAME"), mongo_url=os.getenv("CLOUD_MONGODB_URL"))


Connected: vertical_search_engine_cw | collections: ['crawl_log', 'doc_vectors', 'term_index', 'raw_pages_publications', 'raw_pages_profiles', 'clustered_docs']


In [10]:
print(f"\n{'=' * 20} Clustered data deleting and saving in DB {'=' * 20}\n")

# delete older clustered data
clustered_docs.delete_many({})
print("Deleted older clustered data successfully...")


# converting dataframe into dictionary for saving clustered data into Mongo DB
dict_clustered_docs = df.to_dict(orient="records")

# MongoDB collection in this project requires a unique url field for each document.
for index, record in enumerate(dict_clustered_docs):
    record["url"] = f"clustered-doc-{index}-{record['true_category']}-{record['cluster']}"

# storing document, true_category, cluster, and predicted_category in the DB
clustered_docs.insert_many(dict_clustered_docs)

print(f"\n{'=' * 20} Clustered data stored in the DB with {len(dict_clustered_docs)} length {'=' * 20}\n")



==================== Clustered data deleting and saving in DB ====================

Deleted older clustered data successfully...

==================== Clustered data stored in the DB with 10 length ====================

